# 28. Colab — Smoke Test Básico (Método I y Método II)

Notebook de verificación del pipeline `qml4var` para los dos métodos de pricing
cuántico implementados en el TFM, listo para ejecutarse en Google Colab (GPU).

**Experimento:**
- **Método I** (Supervisado — PDF): arquitectura 6×6, `n_data=500`.
- **Método II** (Semi-supervisado — CDF/IBP): arquitectura 4×4, `n_data=1000`.
- Tres strikes: K ∈ {90, 100, 110} (ITM, ATM, OTM).
- 10 repeticiones por (método, strike).

**La lógica es idéntica a `src/run_experiments.py`:**
- Datos, circuito, closures de pérdida/gradiente, pricing — mismo código.
- Sin guardado en disco: los resultados se acumulan en `results_df` en memoria.

**Selección de métodos:** edita la celda 4 para ejecutar uno o ambos.

---

**Antes de ejecutar:** activa la GPU en Colab
(`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`).

## 0. Verificar GPU del entorno Colab

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("[WARN] nvidia-smi no disponible. Verifica que el entorno tiene GPU activa.")
    print(result.stderr)

## 1. Instalar dependencias

Mismas dependencias que el pipeline principal.
`pennylane-lightning[gpu]` **no es necesario** — se usa `default.qubit` + `backprop`.

In [ ]:
import sys

!{sys.executable} -m pip install --quiet \
    "pennylane>=0.38" \
    "pennylane-lightning>=0.38" \
    numpy scipy

print("Instalación completada.")

## 2. Montar el repositorio

Dos opciones; ejecuta **sólo una** de las dos celdas siguientes.

### Opción A — Google Drive (recomendada si el repo está en Drive)

In [ ]:
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

# Ajusta la ruta si el repo está en otro lugar de tu Drive
REPO_ROOT = "/content/drive/MyDrive/CodeTFM"

sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
print("REPO_ROOT:", REPO_ROOT)

### Opción B — Clonar desde GitHub

In [ ]:
# Sustituye la URL por la de tu repositorio
# !git clone https://github.com/TU_USUARIO/CodeTFM.git /content/CodeTFM

# import os, sys
# REPO_ROOT = "/content/CodeTFM"
# sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
# print("REPO_ROOT:", REPO_ROOT)

## 3. Imports y verificación de GPU en PyTorch

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import torch
from matplotlib.lines import Line2D

from finance import bs_put_price, estimate_price_from_trained_pqc, estimate_price_ibp
from qml4var.adam import adam_optimizer_loop
from qml4var.architectures import hardware_efficient_ansatz, init_weights
from qml4var.data_utils import (
    bs_cdf,
    empirical_cdf,
    generate_method_I_data,
    inverse_rescaling_u_to_xt,
    simulate_black_scholes_data_rescaled,
)
from qml4var.losses import method_I_h1_loss, qdml_loss_workflow, torch_gradient
from qml4var.workflows import mse_workflow, workflow_for_pdf_direct

# --- Verificación CUDA ---
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch version    : {torch.__version__}")
print(f"pennylane version: {qml.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name         : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version     : {torch.version.cuda}")
print(f"TORCH_DEVICE     : {TORCH_DEVICE}")

if TORCH_DEVICE == "cpu":
    print()
    print("[WARN] No se detectó GPU. El experimento continúa en CPU,")
    print("       pero los tiempos serán significativamente mayores.")

## 4. Selección de métodos a ejecutar

Edita las dos variables booleanas para elegir qué métodos ejecutar.
Puedes activar uno, el otro, o ambos.

In [ ]:
# ── Cambia estos valores para controlar qué se ejecuta ──────────────────────
RUN_METHOD_I = True  # Método I  — Supervisado, PDF-Fourier,  arquitectura 6×6
RUN_METHOD_II = True  # Método II — Semi-supervisado, CDF/IBP, arquitectura 4×4
# ─────────────────────────────────────────────────────────────────────────────

active_methods = [m for m, run in [(1, RUN_METHOD_I), (2, RUN_METHOD_II)] if run]
if not active_methods:
    raise ValueError("Activa al menos un método (RUN_METHOD_I o RUN_METHOD_II).")
print(f"Métodos activos: {active_methods}")

## 5. Configuración del experimento

In [ ]:
# ── Black-Scholes ──────────────────────────────────────────────────────────────
BS_S0 = 100.0
BS_R = 0.10
BS_T = 1.0
BS_SIGMA = 0.25

TRAIN_INTERVAL = (-np.pi, np.pi)

# ── Experimento ────────────────────────────────────────────────────────────────
STRIKES = [90, 100, 110]  # ITM, ATM, OTM
REPETITIONS = 10

# ── Hiperparámetros compartidos ────────────────────────────────────────────────
EPOCHS = 300
BETA1 = 0.9
BETA2 = 0.999
PRINT_STEP = 50  # imprime loss cada N épocas
INT_POINTS = 100  # puntos de integración para ∫PDF² en la QDML loss
K_TERMS = 12  # armónicos de Fourier para pricing

# ── Configuración por método (idéntica a METHOD_CONFIGS en run_experiments.py) ──
METHOD_CONFIGS = {
    1: dict(
        name="Method I (Supervisado — PDF)",
        n_qubits=6,
        n_layers=6,
        n_data=500,
        n_test=100,
        lr=0.005,
        alpha_0=0.9,
        alpha_1=0.1,
        base_freq=0.5,
        shift=0.0,
        eval_interval=(-np.pi, np.pi),  # dominio DFT (paper Sec. 3.2.1)
        pricing="pdf_fourier",
    ),
    2: dict(
        name="Method II (Semi-supervisado — CDF/IBP)",
        n_qubits=4,
        n_layers=4,
        n_data=1000,
        n_test=1000,
        lr=0.1,
        alpha_0=0.2,
        alpha_1=0.8,
        base_freq=0.5,
        shift=-np.pi / 2.0,  # rompe paridad par → CDF no degenerada
        pricing="ibp",
    ),
}

total_runs = sum(len(STRIKES) * REPETITIONS for _ in active_methods)
print(f"Strikes     : {STRIKES}")
print(f"Repeticiones: {REPETITIONS}")
print(f"Runs totales: {total_runs}")
for m in active_methods:
    cfg = METHOD_CONFIGS[m]
    print(f"\n  {cfg['name']}")
    print(f"    Arquitectura : {cfg['n_qubits']}×{cfg['n_layers']}")
    print(f"    n_data       : {cfg['n_data']}  |  n_test: {cfg['n_test']}")
    print(f"    epochs={EPOCHS}  lr={cfg['lr']}  α=({cfg['alpha_0']}, {cfg['alpha_1']})")
    print(f"    base_freq={cfg['base_freq']}  shift={cfg['shift']:.4f}")

## 6. Experimento principal

La función `run_experiment` replica fielmente `run_single()` de `src/run_experiments.py`:
datos → circuito → closures de pérdida/gradiente → entrenamiento → pricing.

> **Tiempo estimado** (GPU Tesla T4):
> - Método I 6×6, 300 épocas, n=500: ~4–8 min/run → ~2–4 h para 30 runs.
> - Método II 4×4, 300 épocas, n=1000: ~2–5 min/run → ~1–2.5 h para 30 runs.
> Ajusta `EPOCHS` o `REPETITIONS` si quieres un test más rápido.

In [ ]:
def run_experiment(method_id: int, K: float, rep: int) -> dict:
    """Mirror fiel de run_single() en src/run_experiments.py."""
    cfg = METHOD_CONFIGS[method_id]
    seed = rep * 7919 + int(K) * 97 + cfg["n_data"]

    # ── 1. Datos ──────────────────────────────────────────────────────────────
    if method_id == 1:
        # Método I: grid analítico con etiquetas PDF y su derivada
        train_x, pdf_labels, pdf_deriv_labels, x_min_raw, x_max_raw = generate_method_I_data(
            S0=BS_S0,
            K=K,
            r=BS_R,
            T=BS_T,
            sigma=BS_SIGMA,
            n_points=cfg["n_data"],
            seed=seed,
        )
        # Conjunto test: grid uniforme determinista (sin seed → linspace)
        test_x, test_pdf_labels, _, _, _ = generate_method_I_data(
            S0=BS_S0,
            K=K,
            r=BS_R,
            T=BS_T,
            sigma=BS_SIGMA,
            n_points=cfg["n_test"],
        )
        train_y = pdf_labels.reshape(-1, 1)
    else:
        # Método II: muestras simuladas + CDF empírica
        _, u_t, x_min_raw, x_max_raw = simulate_black_scholes_data_rescaled(
            S0_=BS_S0,
            r_=BS_R,
            T_=BS_T,
            sigma_=BS_SIGMA,
            K_=K,
            n_points=cfg["n_data"],
            seed=seed,
        )
        train_x = u_t
        train_y = empirical_cdf(u_t).reshape(-1, 1)
        # Conjunto test: grid uniforme con CDF analítica BS
        u_test = np.linspace(-np.pi, np.pi, cfg["n_test"]).reshape(-1, 1)
        x_raw_test = inverse_rescaling_u_to_xt(u_test.reshape(-1), x_min_raw, x_max_raw)
        s_t_test = K * np.exp(x_raw_test)
        test_x = u_test
        test_y = bs_cdf(
            s_t_test,
            s_0=BS_S0,
            risk_free_rate=BS_R,
            volatility=BS_SIGMA,
            maturity=BS_T,
        ).reshape(-1, 1)

    # ── 2. Circuito ───────────────────────────────────────────────────────────
    np.random.seed(seed + 42)
    circuit_fn, weights_names, features_names = hardware_efficient_ansatz(
        features_number=1,
        n_qubits_by_feature=cfg["n_qubits"],
        n_layers=cfg["n_layers"],
        base_frecuency=[cfg["base_freq"]],
        shift_feature=[cfg["shift"]],
        torch_device=TORCH_DEVICE,
    )

    # ── 3. workflow_cfg ───────────────────────────────────────────────────────
    workflow_cfg = dict(
        circuit_fn=circuit_fn,
        torch_device=TORCH_DEVICE,
        loss_weights=[cfg["alpha_0"], cfg["alpha_1"]],
        minval=[-np.pi],
        maxval=[np.pi],
        points=INT_POINTS,
        features_names=features_names,
    )

    # ── 4. Closures de pérdida, métrica y gradiente ───────────────────────────
    if method_id == 1:
        _circuit_fn = circuit_fn
        _a0, _a1 = cfg["alpha_0"], cfg["alpha_1"]

        def loss_fn(w):
            w_t = torch.tensor(
                list(w.values()) if isinstance(w, dict) else list(w),
                dtype=torch.float64,
            )
            return method_I_h1_loss(
                w_t,
                train_x,
                pdf_labels,
                pdf_deriv_labels,
                circuit_fn=_circuit_fn,
                device=TORCH_DEVICE,
                alpha_0=_a0,
                alpha_1=_a1,
                create_graph=False,
            ).item()

        def metric_fn(w):
            preds = workflow_for_pdf_direct(w, test_x, **workflow_cfg)["y_predict_pdf"]
            return float(np.mean((preds - test_pdf_labels) ** 2))

        def gradient_fn(w, bx, by):
            def _loss_torch(w_t, bx_, by_):
                return method_I_h1_loss(
                    w_t,
                    bx_,
                    by_,
                    pdf_deriv_labels,
                    circuit_fn=_circuit_fn,
                    device=TORCH_DEVICE,
                    alpha_0=_a0,
                    alpha_1=_a1,
                    create_graph=True,
                )

            return torch_gradient(list(w), bx, by, _loss_torch)

    else:

        def loss_fn(w):
            return qdml_loss_workflow(w, train_x, train_y, **workflow_cfg)

        def metric_fn(w):
            return mse_workflow(w, test_x, test_y, **workflow_cfg)

        def gradient_fn(w, bx, by):
            def _loss_torch(w_t, bx_, by_):
                return qdml_loss_workflow(w_t, bx_, by_, **workflow_cfg)

            return torch_gradient(list(w), bx, by, _loss_torch)

    # ── 5. Entrenamiento ──────────────────────────────────────────────────────
    final_weights = adam_optimizer_loop(
        weights_dict=init_weights(weights_names),
        loss_function=loss_fn,
        metric_function=metric_fn,
        gradient_function=gradient_fn,
        batch_generator=[(train_x, train_y)],
        initial_time=0,
        epochs=EPOCHS,
        learning_rate=cfg["lr"],
        beta1=BETA1,
        beta2=BETA2,
        tolerance=-1e30,
        n_counts_tolerance=EPOCHS + 1,
        print_step=PRINT_STEP,
        progress_bar=True,
        progress_desc=f"M{method_id} K={int(K)} rep={rep}",
        progress_leave=False,
    )

    # ── 6. Pricing ────────────────────────────────────────────────────────────
    artifacts = {"workflow_cfg": workflow_cfg}
    base_kwargs = dict(
        weights=final_weights,
        artifacts=artifacts,
        K_=K,
        x_min_raw=x_min_raw,
        x_max_raw=x_max_raw,
        train_interval=TRAIN_INTERVAL,
        risk_free_rate=BS_R,
        delta_t=BS_T,
        k_terms=K_TERMS,
    )
    try:
        if cfg["pricing"] == "pdf_fourier":
            est_price = float(
                estimate_price_from_trained_pqc(
                    **base_kwargs,
                    eval_interval=cfg["eval_interval"],
                )
            )
        else:
            est_price = float(estimate_price_ibp(**base_kwargs))
    except Exception as exc:
        print(f"  [pricing error] {exc}")
        est_price = float("nan")

    bs_price = bs_put_price(S0_=BS_S0, K_=K, r_=BS_R, sigma_=BS_SIGMA, T_=BS_T)
    final_mse = metric_fn(final_weights)
    abs_err = abs(est_price - bs_price) if np.isfinite(est_price) else float("nan")
    rel_err = (abs_err / bs_price) if (np.isfinite(abs_err) and bs_price != 0) else float("nan")

    return dict(
        method=method_id,
        n_qubits=cfg["n_qubits"],
        n_layers=cfg["n_layers"],
        n_data=cfg["n_data"],
        K=int(K),
        rep=rep,
        seed=seed,
        bs_price=bs_price,
        estimated_price=est_price,
        abs_error=abs_err,
        rel_error=rel_err,
        final_mse=final_mse,
    )


# ── Loop principal ────────────────────────────────────────────────────────────
results = []
global_t0 = perf_counter()
run_idx = 0

for method_id in active_methods:
    for K in STRIKES:
        for rep in range(1, REPETITIONS + 1):
            run_idx += 1
            t0 = perf_counter()

            result = run_experiment(method_id, K, rep)

            t1 = perf_counter()
            elapsed = t1 - global_t0
            eta_min = (total_runs - run_idx) * (elapsed / run_idx) / 60

            print(
                f"[{run_idx:>2}/{total_runs}]  M{method_id}  K={K:<3}  rep={rep:<2}  "
                f"BS={result['bs_price']:.4f}  Est={result['estimated_price']:.4f}  "
                f"AbsErr={result['abs_error']:.4f}  MSE={result['final_mse']:.2e}  "
                f"t={t1 - t0:.0f}s  ETA={eta_min:.1f}min"
            )
            results.append(result)

results_df = pd.DataFrame(results)
print(f"\nTiempo total: {(perf_counter() - global_t0) / 60:.1f} min")
results_df

## 7. Resultados numéricos

In [ ]:
if results_df.empty:
    print("[INFO] No hay resultados. Ejecuta primero el experimento (Sección 6).")
else:
    summary = results_df.groupby(["method", "K"], as_index=False).agg(
        method_name=("method", lambda x: METHOD_CONFIGS[x.iloc[0]]["name"]),
        bs_price=("bs_price", "first"),
        mean_price=("estimated_price", "mean"),
        std_price=("estimated_price", "std"),
        mean_abs_err=("abs_error", "mean"),
        mean_rel_err=("rel_error", "mean"),
        mean_mse=("final_mse", "mean"),
    )
    summary["std_price"] = summary["std_price"].fillna(0.0)
    summary["rel_error_%"] = (summary["mean_rel_err"] * 100).round(2)

    display_cols = [
        "method_name",
        "K",
        "bs_price",
        "mean_price",
        "std_price",
        "mean_abs_err",
        "rel_error_%",
        "mean_mse",
    ]
    print(summary[display_cols].to_string(index=False))

    print("\n─── Resumen estadístico por método ───")
    for m in active_methods:
        sub = results_df[results_df["method"] == m]
        print(f"\n{METHOD_CONFIGS[m]['name']}")
        print(f"  Runs              : {len(sub)}")
        print(f"  Error relativo")
        print(f"    Media           : {sub['rel_error'].mean():.2%}")
        print(f"    Mediana         : {sub['rel_error'].median():.2%}")
        print(f"    Desv. estándar  : {sub['rel_error'].std():.2%}")
        print(f"    P10 / P90       : {sub['rel_error'].quantile(0.1):.2%} / {sub['rel_error'].quantile(0.9):.2%}")
        print(f"  Error absoluto")
        print(f"    Media           : {sub['abs_error'].mean():.4f}")
        print(f"    Mediana         : {sub['abs_error'].median():.4f}")
        print(f"  MSE final mediano : {sub['final_mse'].median():.2e}")

## 8. Figura 5 — Precios estimados vs. Black-Scholes

Adaptación de la Figura 5 del artículo (`04_analisis_resultados.ipynb`, celda 30).
Un panel por método activo; cada serie corresponde a un strike K.
- Punto con barra de error: media ± desv. estándar sobre las 10 repeticiones.
- Línea discontinua del mismo color: precio analítico Black-Scholes.

In [ ]:
if results_df.empty:
    print("[INFO] No hay resultados. Ejecuta primero el experimento (Sección 6).")
else:
    K_COLORS = {90: "#2196F3", 100: "#FF9800", 110: "#4CAF50"}
    K_LABELS = {90: r"$K=90$", 100: r"$K=100$", 110: r"$K=110$"}

    n_panels = len(active_methods)
    fig, axes = plt.subplots(1, n_panels, figsize=(6.5 * n_panels, 5.2), sharey=False)
    if n_panels == 1:
        axes = [axes]

    for ax, method_id in zip(axes, active_methods):
        cfg = METHOD_CONFIGS[method_id]
        sub = results_df[results_df["method"] == method_id]
        sizes = sorted(sub["n_data"].unique())

        for k in STRIKES:
            color = K_COLORS[k]
            ksub = sub[sub["K"] == k]
            grp = ksub.groupby("n_data")["estimated_price"]
            means = grp.mean()
            stds = grp.std().fillna(0.0)

            ax.errorbar(
                means.index.values,
                means.values,
                yerr=stds.values,
                fmt="o-",
                color=color,
                capsize=4,
                capthick=1.2,
                lw=1.5,
                markersize=6,
                label=K_LABELS[k],
            )
            bs_val = float(ksub["bs_price"].iloc[0])
            ax.axhline(bs_val, color=color, linestyle="--", lw=1.2, alpha=0.85)

        arch_str = f"({cfg['n_qubits']}×{cfg['n_layers']})"
        ax.set_title(
            f"Estimated prices — {cfg['name']}\n{arch_str}",
            fontsize=10,
            fontweight="bold",
        )
        ax.set_xlabel("Size of the dataset (n)", fontsize=9)
        ax.set_ylabel("Price", fontsize=9)
        ax.set_xticks(sizes)
        ax.set_xticklabels([str(s) for s in sizes], fontsize=9)
        ax.grid(True, alpha=0.25, linestyle="--")

    handles = [Line2D([0], [0], color=K_COLORS[k], marker="o", lw=1.5, label=K_LABELS[k]) for k in STRIKES]
    handles.append(Line2D([0], [0], color="gray", linestyle="--", lw=1.2, label="Black-Scholes (exacto)"))
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=4,
        fontsize=9,
        bbox_to_anchor=(0.5, -0.05),
        frameon=True,
    )
    fig.suptitle(
        "Figura 5: Precios estimados vs. Black-Scholes por método y arquitectura",
        fontweight="bold",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

## 9. Checklist final

| Check | Esperado |
|-------|----------|
| `nvidia-smi` imprime GPU | Sin errores |
| `TORCH_DEVICE = cuda` | `cuda` (no `cpu`) |
| Loop sin excepciones | 30 runs por método activo (3 strikes × 10 reps) |
| Precios estimados | Valores finitos (no NaN/inf) |
| Error relativo | Decrece con el entrenamiento |
| Gráfica renderizada | Paneles con puntos y líneas discontinuas BS |

Si todos los checks pasan, la configuración está lista para los experimentos
completos con `src/run_experiments.py`.